### K-Nearest Neighbors (KNN) Classifier: Machine Learning Theory & Implementation

The **K-Nearest Neighbors (KNN)** algorithm is a simple, intuitive, and versatile non-parametric supervised learning algorithm. It can be used for both **classification** and **regression** tasks, though it is most famously applied to classification. 

Unlike most algorithms, KNN is a **lazy learner**, meaning it does not explicitly learn a model during a training phase. Instead, it memorizes the entire training dataset and does all its heavy lifting during the prediction phase.

---

#### 1. The Core Intuition: "Birds of a Feather Flock Together"

The core philosophy of KNN is that data points with similar attributes tend to belong to the same class or have similar target values. 

When you give KNN a new, unseen data point (a query point), it:
1. Looks at the entire training dataset.
2. Finds the **$K$** closest data points to that query point.
3. Conducts a "majority vote" (for classification) or calculates the average (for regression) among those $K$ neighbors to determine the final output.



---

#### 2. The Mathematics of "Closeness" (Distance Metrics)

To determine which points are the "nearest," KNN relies on distance metrics calculated in a multi-dimensional feature space. Let $x$ and $y$ be two data points with $n$ features.

##### A. Euclidean Distance
The most common metric. It calculates the straight-line distance between two points (derived from the Pythagorean theorem).

$$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$

We calculate the hypotaneous distance.

##### B. Manhattan Distance
Calculates the distance if you could only travel along grid lines (like walking the blocks of Manhattan).

$$d(x, y) = \sum_{i=1}^{n} |x_i - y_i|$$

So WE do not calculate the Hypotaneous here rater (base + perpendicular).

##### C. Minkowski Distance
A generalized format of both Euclidean and Manhattan distances. 

$$d(x, y) = \left( \sum_{i=1}^{n} |x_i - y_i|^p \right)^{\frac{1}{p}}$$

* When $p = 1$, it becomes **Manhattan Distance**.
* When $p = 2$, it becomes **Euclidean Distance**.

---

#### 3. Choosing the Right $K$ Value

The hyperparameter **$K$** represents the number of neighbors the algorithm considers. Choosing the right $K$ is a crucial balancing act:

* **If $K$ is too small (e.g., $K = 1$):** The model is highly sensitive to noise and outliers in the data. This leads to **overfitting** (low bias, high variance).
* **If $K$ is too large (e.g., $K = 100$):** The boundaries between classes become blurred. The model starts pulling in points from other classes, leading to **underfitting** (high bias, low variance).

> **Rule of Thumb:** A common starting point is to set $K = \sqrt{N}$, where $N$ is the total number of training samples. If you are doing binary classification, it is best to choose an **odd number** for $K$ to avoid tie votes.

---

#### 4. Pros and Cons

### **The Good**
* **Simple and Intuitive:** Easy to understand, explain, and implement.
* **No Assumptions:** It makes no assumptions about the underlying distribution of the data (unlike Linear Regression or Naive Bayes).
* **Naturally Multiclass:** Easily handles datasets with more than two target classes.

### **The Bad**
* **Computationally Expensive:** Because it calculates the distance to *every single point* in the dataset at test time, it becomes incredibly slow as your dataset grows ($O(N \cdot M)$ complexity).
* **High Memory Usage:** It requires storing the entire dataset in RAM.
* **Sensitive to Scale:** Features with larger numeric ranges (like "Salary" vs "Age") will completely dominate the distance calculations. **Feature scaling (Normalization/Standardization) is mandatory for KNN.**


### KNN Optimization: KD-Tree and Ball Tree Algorithms

By default, a standard K-Nearest Neighbors (KNN) classifier uses a **Brute-Force** approach. For a dataset with $N$ samples and $D$ features, it calculates the distance to *every single point* in the training set to find the nearest neighbors, resulting in a prediction time complexity of $\mathcal{O}(N \cdot D)$. 

As $N$ grows into millions, brute force becomes computationally impossible. To optimize this, we use spatial data structures like **KD-Trees** and **Ball Trees** to index our data and reduce prediction time to $\mathcal{O}(D \cdot \log N)$.

---

#### 1. KD-Tree (K-Dimensional Tree)

A **KD-Tree** is a space-partitioning `binary-tree` data structure that recursively bi-sections the feature space along axis-aligned hyperplanes.

##### How it Works:
1. **Choose an Axis:** Select a feature dimension (e.g., alternating between the $x$ and $y$ axis).
2. **Split at the Median:** Find the median value of the data points along that axis and split the space into two halves (left and right child nodes).
3. **Repeat:** Recursively repeat the process for the child nodes using the next feature dimension until a stopping condition is met.



##### Limitation: The Curse of Dimensionality
KD-Trees are highly efficient in low-dimensional spaces. However, because they must split purely along coordinate axes, as the number of features ($D$) grows ($D > 20$), the algorithm ends up needing to search almost every branch anyway. In high dimensions, its performance degrades back to a brute-force search $\mathcal{O}(N)$.

---

#### 2. Ball Tree

To overcome the high-dimensional limitations of KD-Trees, the **Ball Tree** algorithm partitions data into a series of nesting n-dimensional hyperspheres (balls) rather than axis-aligned boxes.

##### How it Works:
1. **Define a Centroid:** Choose a central point that splits the current data into two clusters.
2. **Create a Hypersphere:** Draw a bounding sphere around each cluster defined by a centroid and a radius $R$ that contains all points in that cluster.
3. **Repeat:** Recursively create smaller, nested hyperspheres within the larger ones.



During a search, if the distance from the query point to the surface of a "ball" is greater than the distance to the current closest neighbor, the algorithm can instantly skip searching the *entire* ball and all its internal sub-trees. This makes it far more robust in higher dimensions than a KD-Tree.

---

#### 3. Direct Comparison

| Feature / Metric | Brute-Force | KD-Tree | Ball Tree |
| :--- | :--- | :--- | :--- |
| **Query Time Complexity** | $\mathcal{O}(N \cdot D)$ | $\mathcal{O}(D \cdot \log N)$ | $\mathcal{O}(D \cdot \log N)$ |
| **Space Partition Shape** | None | Hyperrectangles (Axis-Aligned) | Hyperspheres (Centroid-Based) |
| **Optimal Dimensionality** | Any | Low dimensions ($D < 20$) | High dimensions ($D > 20$) |
| **Training/Build Time** | $\mathcal{O}(1)$ (Instant) | $\mathcal{O}(D \cdot N \log N)$ | $\mathcal{O}(D \cdot N \log N)$ |

---

## ##4. Python Implementation via `scikit-learn`

`scikit-learn` allows you to manually select these optimization structures using the `algorithm` hyperparameter, or use `'auto'` to let the library analyze your data size and features to choose the best one.

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier
import time

# 1. Generate a large synthetic dataset
X, y = make_classification(n_samples=20000, n_features=10, random_state=42)

# 2. Define models utilizing different optimization structures
models = {
    "Brute-Force": KNeighborsClassifier(n_neighbors=5, algorithm='brute'),
    "KD-Tree": KNeighborsClassifier(n_neighbors=5, algorithm='kd_tree'),
    "Ball Tree": KNeighborsClassifier(n_neighbors=5, algorithm='ball_tree')
}

# 3. Benchmark training and prediction times
for name, model in models.items():
    # Fit (Build the tree structure)
    start_train = time.time()
    model.fit(X, y)
    train_time = time.time() - start_train
    
    # Predict (Query the tree structure)
    start_pred = time.time()
    _ = model.predict(X[:1000]) # Querying first 1000 samples
    pred_time = time.time() - start_pred
    
    print(f"=== {name} ===")
    print(f"Tree Build Time: {train_time:.5f} seconds")
    print(f"Query Time (1k pts): {pred_time:.5f} seconds\n")